# Camada Silver — `notas`

**Disciplina:** Mineração de Dados — FATEC Matão-SP (Prof. Rodrigo Vidrich)  
**Projeto:** Trabalho Final (P2) — Arquitetura Medallion (`Bronze → Silver → Gold → Power BI`)  
**Integrantes:** Jerônimo Barbieri Junior, Luan Chierato Oliveira, Miguel Miranda Morandini, Victor Hugo Malipense Testi  
**Data:** 13/11/2026  

**Técnicas aplicadas:** Conversão de Tipos (`id_aluno`, `data`, `nota`), Limpeza Textual (`disciplina`), Validação de Regra de Negócio (`nota` entre 0 e 10 preservando `nota_bruta` e `flag_nota_invalida`), Detecção de Outliers (IQR e Z-Score) e Remoção de Duplicatas.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "notebooks" / "utils").exists():
        if str(candidate / "notebooks") not in sys.path:
            sys.path.insert(0, str(candidate / "notebooks"))
        break

from utils.silver_utils import (
    anonymize_name_lgpd,
    detect_outliers_iqr,
    detect_outliers_zscore,
    fix_mojibake_text,
    get_silver_paths,
    load_bronze,
    normalize_id_aluno,
    parse_br_float,
    parse_date_col,
    save_silver_outputs,
)

ROOT_DIR, BRONZE_DIR, SILVER_DIR = get_silver_paths()

In [ ]:
df = load_bronze("notas", BRONZE_DIR)
rows_in = len(df)

df["id_aluno"] = normalize_id_aluno(df["id_aluno"])
df["disciplina"], mojibake_disc = fix_mojibake_text(df["disciplina"])
df["disciplina"] = df["disciplina"].str.title().astype("category")
df["data"] = parse_date_col(df["data"])

df["nota_bruta"] = parse_br_float(df["nota"])
invalida_mask = df["nota_bruta"].isna() | (df["nota_bruta"] < 0) | (df["nota_bruta"] > 10)
df["flag_nota_invalida"] = invalida_mask
# Mantém NaN nas inválidas para não distorcer médias acadêmicas
df["nota"] = df["nota_bruta"].where(~invalida_mask, np.nan)

df["situacao_nota"] = np.where(
    df["flag_nota_invalida"],
    "Inválida (Fora 0-10)",
    np.where(df["nota"] >= 6.0, "Aprovado (>=6)", "Abaixo da Média (<6)"),
)
df["situacao_nota"] = df["situacao_nota"].astype("category")

df["flag_nota_outlier_iqr"], iqr_info = detect_outliers_iqr(df["nota"])
df["flag_nota_outlier_zscore"], z_count = detect_outliers_zscore(df["nota"])

dups_removed = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

log = {
    "dataset": "notas",
    "rows_in": rows_in,
    "rows_out": int(len(df)),
    "duplicates_removed": dups_removed,
    "mojibake_fixed_disciplina": mojibake_disc,
    "notas_invalidas_anuladas": int(invalida_mask.sum()),
    "nota_media_valida": round(float(df["nota"].mean()), 4),
    "nota_mediana_valida": round(float(df["nota"].median()), 4),
    "outliers_iqr": iqr_info,
    "outliers_zscore": z_count,
}
save_silver_outputs(df, log, "notas", SILVER_DIR)
print(json.dumps(log, ensure_ascii=False, indent=2))
df.head(5)